In [3]:
import sqlite3

conn = sqlite3.connect("ALMS.db")
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

# -------------------
# Tabellen erstellen
# -------------------
cur.executescript("""
DROP TABLE IF EXISTS rennergebnis;
DROP TABLE IF EXISTS fahrzeuge;
DROP TABLE IF EXISTS fahrer;
DROP TABLE IF EXISTS rennen;
DROP TABLE IF EXISTS team;

CREATE TABLE team (
    team_id       INTEGER PRIMARY KEY,
    team_name     TEXT NOT NULL UNIQUE,
    teamchef      TEXT,
    herkunftsland TEXT
);

CREATE TABLE fahrer (
    fahrer_id     INTEGER PRIMARY KEY,
    team_id       INTEGER NOT NULL,
    fahrername    TEXT NOT NULL,
    geburtsdatum  TEXT,
    nationalitaet TEXT,
    startnummer   INTEGER NOT NULL,
    FOREIGN KEY (team_id) REFERENCES team(team_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    UNIQUE(team_id, startnummer)
);

CREATE TABLE fahrzeuge (
    fahrzeug_id INTEGER PRIMARY KEY,
    team_id     INTEGER NOT NULL,
    hersteller  TEXT NOT NULL,
    baujahr     INTEGER,
    fahrklasse  TEXT NOT NULL,
    FOREIGN KEY (team_id) REFERENCES team(team_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    CHECK (baujahr IS NULL OR (baujahr BETWEEN 1950 AND 2100))
);

CREATE TABLE rennen (
    renn_id   INTEGER PRIMARY KEY,
    datum     TEXT NOT NULL,
    rennzeit  INTEGER,
    strecke   TEXT NOT NULL
);

CREATE TABLE rennergebnis (
    ergebnis_id INTEGER PRIMARY KEY,
    fahrer_id   INTEGER NOT NULL,
    fahrzeug_id INTEGER NOT NULL,
    renn_id     INTEGER NOT NULL,
    platzierung INTEGER NOT NULL CHECK (platzierung >= 1),
    punkte      INTEGER NOT NULL CHECK (punkte >= 0),

    FOREIGN KEY (fahrer_id) REFERENCES fahrer(fahrer_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    FOREIGN KEY (fahrzeug_id) REFERENCES fahrzeuge(fahrzeug_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    FOREIGN KEY (renn_id) REFERENCES rennen(renn_id)
        ON UPDATE CASCADE
        ON DELETE CASCADE,

    UNIQUE(fahrer_id, renn_id)
);
""")

# -------------------
# Dummy-Daten
# -------------------

# Teams (+ Dodge neu)
cur.executemany(
    "INSERT INTO team (team_id, team_name, teamchef, herkunftsland) VALUES (?, ?, ?, ?)",
    [
        (1, "Alpine Velocity Racing", "M. Berger", "Austria"),
        (2, "Nordic Apex Motorsport", "E. Lindström", "Sweden"),
        (3, "Rhein Motorsport Works", "T. Klein", "Germany"),
        (4, "Pacific Horizon Racing", "S. Tanaka", "Japan"),
        (5, "Dodge Thunder Motorsport", "R. Coleman", "USA"),
    ]
)

# Fahrzeuge (+ Dodge neu)
cur.executemany(
    "INSERT INTO fahrzeuge (fahrzeug_id, team_id, hersteller, baujahr, fahrklasse) VALUES (?, ?, ?, ?, ?)",
    [
        (1, 1, "Porsche", 2025, "GTP"),
        (2, 2, "Cadillac", 2025, "GTP"),
        (3, 3, "BMW", 2025, "GTD Pro"),
        (4, 4, "Acura", 2025, "LMP2"),
        (5, 5, "Dodge", 2025, "GTD Pro"),
    ]
)

# Fahrer
# - Muhammet Altuntas (Österreich) bei Dodge (guter Fahrer)
# - Emir Alici (Indien) bei BMW Team (immer letzter)
cur.executemany(
    """INSERT INTO fahrer (fahrer_id, team_id, fahrername, geburtsdatum, nationalitaet, startnummer)
       VALUES (?, ?, ?, ?, ?, ?)""",
    [
        (1, 1, "Lena Hartmann", "2001-04-12", "Austria", 7),
        (2, 1, "Jonas Gruber", "1998-09-30", "Austria", 77),

        (3, 2, "Erik Nyström", "1997-02-18", "Sweden", 10),
        (4, 2, "Maja Holm", "2000-06-03", "Sweden", 110),

        (5, 3, "Tobias Keller", "1996-11-21", "Germany", 24),
        (6, 3, "Miriam Vogt", "1999-08-07", "Germany", 124),
        (9, 3, "Emir Alici", "2001-05-05", "India", 999),   # immer letzter

        (7, 4, "Sora Tanaka", "1995-01-14", "Japan", 55),
        (8, 4, "Hana Ito", "2002-12-28", "Japan", 155),

        (10, 5, "Muhammet Altuntas", "1999-03-02", "Austria", 33),  # guter Fahrer
    ]
)

# Rennen (5 Rennen)
cur.executemany(
    "INSERT INTO rennen (renn_id, datum, rennzeit, strecke) VALUES (?, ?, ?, ?)",
    [
        (1, "2025-01-26", 1440, "Daytona International Speedway"),
        (2, "2025-03-15",  720, "Sebring International Raceway"),
        (3, "2025-04-12",  600, "Long Beach Street Circuit"),
        (4, "2025-06-22",  360, "Watkins Glen International"),
        (5, "2025-10-11",  600, "Road Atlanta"),
    ]
)

# -------------------
# Ergebnisse
# Punkte: wir nehmen ein einfaches Schema:
# Platz 1..8 = 25,18,15,12,10,8,6,4
# Platz 9..10 = 0 Punkte (damit Emir immer letzter sein kann)
# -------------------

def points_for_place(place: int) -> int:
    table = {1:25, 2:18, 3:15, 4:12, 5:10, 6:8, 7:6, 8:4}
    return table.get(place, 0)

# Pro Rennen 10 Platzierungen (10 Fahrer)
# Wir lassen Muhammet oft vorne (1-2) und Emir immer Platz 10.
placements_by_race = {
    1: [  # Daytona
        (10, 1), (3, 2), (1, 3), (5, 4), (7, 5),
        (4, 6), (2, 7), (6, 8), (8, 9), (9,10)
    ],
    2: [  # Sebring
        (10, 1), (1, 2), (3, 3), (7, 4), (5, 5),
        (2, 6), (4, 7), (8, 8), (6, 9), (9,10)
    ],
    3: [  # Long Beach
        (5, 1), (10, 2), (1, 3), (3, 4), (7, 5),
        (6, 6), (2, 7), (4, 8), (8, 9), (9,10)
    ],
    4: [  # Watkins Glen
        (10, 1), (7, 2), (3, 3), (1, 4), (5, 5),
        (8, 6), (2, 7), (4, 8), (6, 9), (9,10)
    ],
    5: [  # Road Atlanta
        (3, 1), (10, 2), (7, 3), (1, 4), (5, 5),
        (4, 6), (8, 7), (2, 8), (6, 9), (9,10)
    ],
}

ergebnis_id = 1
rows = []

for renn_id, pairs in placements_by_race.items():
    for fahrer_id, place in pairs:
        # Fahrzeug = Fahrzeug vom Team des Fahrers
        cur.execute("SELECT team_id FROM fahrer WHERE fahrer_id = ?", (fahrer_id,))
        team_id = cur.fetchone()[0]
        cur.execute("SELECT fahrzeug_id FROM fahrzeuge WHERE team_id = ?", (team_id,))
        fahrzeug_id = cur.fetchone()[0]

        rows.append((
            ergebnis_id,
            fahrer_id,
            fahrzeug_id,
            renn_id,
            place,
            points_for_place(place)
        ))
        ergebnis_id += 1

cur.executemany(
    """INSERT INTO rennergebnis
       (ergebnis_id, fahrer_id, fahrzeug_id, renn_id, platzierung, punkte)
       VALUES (?, ?, ?, ?, ?, ?)""",
    rows
)

conn.commit()

# -------------------
# Checks / Ausgaben
# -------------------
print("Saisonwertung (Fahrer):")
cur.execute("""
SELECT f.fahrername, f.nationalitaet, SUM(e.punkte) AS gesamtpunkte
FROM rennergebnis e
JOIN fahrer f ON f.fahrer_id = e.fahrer_id
GROUP BY f.fahrer_id
ORDER BY gesamtpunkte DESC;
""")
for name, nat, pts in cur.fetchall():
    print(f"- {name} ({nat}): {pts} Punkte")

print("\nEmir Alici Ergebnisse (soll immer letzter sein):")
cur.execute("""
SELECT r.strecke, e.platzierung, e.punkte
FROM rennergebnis e
JOIN rennen r ON r.renn_id = e.renn_id
WHERE e.fahrer_id = 9
ORDER BY r.renn_id;
""")
for strecke, platz, punkte in cur.fetchall():
    print(f"- {strecke}: Platz {platz}, Punkte {punkte}")

conn.close()

Saisonwertung (Fahrer):
- Muhammet Altuntas (Austria): 111 Punkte
- Erik Nyström (Sweden): 85 Punkte
- Lena Hartmann (Austria): 72 Punkte
- Tobias Keller (Germany): 67 Punkte
- Sora Tanaka (Japan): 65 Punkte
- Jonas Gruber (Austria): 30 Punkte
- Maja Holm (Sweden): 30 Punkte
- Hana Ito (Japan): 18 Punkte
- Miriam Vogt (Germany): 12 Punkte
- Emir Alici (India): 0 Punkte

Emir Alici Ergebnisse (soll immer letzter sein):
- Daytona International Speedway: Platz 10, Punkte 0
- Sebring International Raceway: Platz 10, Punkte 0
- Long Beach Street Circuit: Platz 10, Punkte 0
- Watkins Glen International: Platz 10, Punkte 0
- Road Atlanta: Platz 10, Punkte 0
